In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# ==========================
# 1. Load Dataset
# ==========================
df = pd.read_csv("crop_yield_1.csv")


# Rename columns
df = df.rename(columns={
    "Temperature (°C)": "temperature",
    "Rainfall (mm)": "rainfall",
    "Humidity (%)": "humidity",
    "Soil Type": "soil_type",
    "Weather Condition": "weather_condition",
    "Crop Type": "crop_type",
    "Yield (tons/hectare)": "yield"
})

# Check for missing values
df = df.dropna()

# Log transform rainfall to handle skewness
df["rainfall"] = np.log1p(df["rainfall"])

# Features & Target
X = df.drop(columns=["yield"])
y = df["yield"]

# ==========================
# 2. Train-Test Split
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================
# 3. Feature Engineering
# ==========================
numeric_features = ["temperature", "humidity", "rainfall"]
categorical_features = ["soil_type", "crop_type", "weather_condition"]

# Numeric Transformer: Standardize + Polynomial Features
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False))
])

# Categorical Transformer: OneHotEncoding
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combine Numeric & Categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# ==========================
# 4. Model Pipeline
# ==========================
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

# ==========================
# 5. Hyperparameter Tuning
# ==========================
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    scoring="neg_mean_squared_error"
)

grid_search.fit(X_train, y_train)

# Best model
best_pipeline = grid_search.best_estimator_

# ==========================
# 6. Model Evaluation
# ==========================
y_pred = best_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("✅ Best Parameters:", grid_search.best_params_)
print("📉 MSE:", mse)
print("📈 R² Score:", r2)

# ==========================
# 7. Save Model
# ==========================
joblib.dump(best_pipeline, "yield_model_1.pkl")
print("💾 Model saved to models/yield_model.pkl")
